# Student 1 — Learning-Style Score Integration + LSTM

**Task:** Compute each learner's FSLSM Visual–Verbal learning-style score, integrate it as a
*static feature* alongside the *sequential* clickstream data (source/event/category token
streams + inter-event time gaps), train an LSTM (BiLSTM) dropout-prediction model on the
fused representation, and compare it against the sequential-only baseline from
`03_Model_Training_Evaluation.ipynb`.

**Design choices (so results stay comparable across the team's experiments):**

1. **Same raw data, same 2-week window, same train/test split** as `02_Data_Preprocessing.ipynb`
   / `03_Model_Training_Evaluation.ipynb`. Nothing about the split or filtering is changed here.
2. **FSLSM score is computed on the identical first-2-week window** that the sequence model
   sees (not the full course log). This keeps the static feature and the sequence feature
   describing the *same* slice of behavior, and avoids leaking post-window activity into a
   "static" feature that is supposed to summarize the same interval the LSTM reads.
3. **Formula follows the PDF exactly**: `V = video + access`, `R = wiki + discussion`,
   `S = (V - R) / (V + R)`, tie (`V == R`, including `V == R == 0`) → `Verbal`.
4. **Fusion for Student 1 is a simple, well-understood one** — concatenate a small static
   MLP embedding with the BiLSTM's final hidden state before the classifier head (a form of
   intermediate/late fusion). Student 3 is exploring early/late/attention-based fusion
   variants in more depth; this notebook exists to establish the FSLSM-augmented LSTM result
   and the apples-to-apples comparison against the baseline, not to exhaust fusion design space.
5. Both the **baseline (sequential-only)** and the **FSLSM-augmented** model are trained in
   this same notebook, with the same hyperparameters (3 epochs, Adam lr=1e-3, batch size 128,
   length-bucketed batching, BCE loss) so the comparison isolates the effect of adding the
   static feature.

Outputs: trained metrics for both models, a comparison table, and the FSLSM feature tables
saved to `data/processed/` / results saved to `data/results/` for the rest of the team
(Student 2's Transformer notebook and Student 3's fusion-comparison notebook) to reuse.

## 1. Load raw data & replicate the shared preprocessing

This mirrors `02_Data_Preprocessing.ipynb` exactly (first-2-weeks filter, category merge,
token construction, sequence building, vocab encoding) so the sequential branch of this
model sees precisely the same inputs the baseline model does.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import os

print("Loading raw data...")
log_train = pd.read_csv("data/raw/train/train/log_train.csv")
log_test = pd.read_csv("data/raw/test/test/log_test.csv")
object_df = pd.read_csv("data/raw/object.csv")
truth_train = pd.read_csv("data/raw/train/train/truth_train.csv", header=None, names=["student_id", "dropout"])
truth_test = pd.read_csv("data/raw/test/test/truth_test.csv", header=None, names=["student_id", "dropout"])

log_train['time'] = pd.to_datetime(log_train['time'])
log_test['time'] = pd.to_datetime(log_test['time'])

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# ==========================================
# FILTER FIRST 2 WEEKS  (identical to 02_Data_Preprocessing.ipynb)
# ==========================================
def filter_first_2_weeks(df):
    first_time = df.groupby("enrollment_id")["time"].transform("min")
    return df[(df["time"] - first_time).dt.total_seconds() <= 14 * 24 * 3600].copy()

log_train = filter_first_2_weeks(log_train)
log_test = filter_first_2_weeks(log_test)

print(f"log_train (first 2 weeks): {log_train.shape}, log_test: {log_test.shape}")

In [ ]:
# ==========================================
# MERGE CATEGORY  (identical to 02_Data_Preprocessing.ipynb)
# ==========================================
log_train = log_train.merge(object_df[['module_id', 'category']], how='left', left_on='object', right_on='module_id')
log_test = log_test.merge(object_df[['module_id', 'category']], how='left', left_on='object', right_on='module_id')

log_train['category'] = log_train['category'].fillna("unknown")
log_test['category'] = log_test['category'].fillna("unknown")

## 2. FSLSM static feature: Binary Visual–Verbal score

Computed strictly per the PDF spec, from the same first-2-week logs loaded above (i.e. the
same rows that feed the token sequences below — no extra data is used for the static
feature that the sequential branch does not also see).

In [ ]:
# ==========================================
# FSLSM BINARY VISUAL-VERBAL SCORE  (per Binary_Visual_Verbal_Scoring.pdf)
#
#   V_i = Video_i + Access_i
#   R_i = Wiki_i  + Discussion_i
#   S_i = (V_i - R_i) / (V_i + R_i)          in [-1, 1]
#   Label_i = Visual  if V_i > R_i
#           = Verbal  if V_i < R_i
#           = Verbal  if V_i == R_i  (tie-break rule from the spec, applied to V_i == R_i == 0 too)
# ==========================================
FSLSM_VISUAL_EVENTS = ['video', 'access']
FSLSM_VERBAL_EVENTS = ['wiki', 'discussion']

def compute_fslsm_features(df):
    # df: filtered event log (enrollment_id, event, ...). Returns one row per
    # enrollment_id with V, R, S and a binary visual_verbal_label (1=Visual, 0=Verbal).
    counts = df.groupby(['enrollment_id', 'event']).size().unstack(fill_value=0)
    for col in FSLSM_VISUAL_EVENTS + FSLSM_VERBAL_EVENTS:
        if col not in counts.columns:
            counts[col] = 0

    V = counts[FSLSM_VISUAL_EVENTS].sum(axis=1)
    R = counts[FSLSM_VERBAL_EVENTS].sum(axis=1)
    total = V + R
    S = np.where(total > 0, (V - R) / total, 0.0)
    label = np.where(V > R, 1, 0)  # 1 = Visual, 0 = Verbal (covers the V==R tie-break)

    out = pd.DataFrame({
        'student_id': counts.index,
        'V': V.values,
        'R': R.values,
        'S': S,
        'visual_verbal_label': label
    }).reset_index(drop=True)
    return out

fslsm_train = compute_fslsm_features(log_train)
fslsm_test = compute_fslsm_features(log_test)

print("FSLSM feature summary (train):")
print(fslsm_train[['V', 'R', 'S', 'visual_verbal_label']].describe())
print(f"\nVisual/Verbal split (train): {fslsm_train['visual_verbal_label'].value_counts().to_dict()}")

os.makedirs("data/processed", exist_ok=True)
fslsm_train.to_csv("data/processed/fslsm_train.csv", index=False)
fslsm_test.to_csv("data/processed/fslsm_test.csv", index=False)
print("\nSaved FSLSM feature tables to data/processed/fslsm_train.csv / fslsm_test.csv (for Student 2 & 3 to reuse).")

## 3. Sequential feature engineering & sequence building

Identical to `02_Data_Preprocessing.ipynb`.

In [ ]:
# ==========================================
# FEATURE ENGINEERING  (identical to 02_Data_Preprocessing.ipynb)
# ==========================================
def process_features(df):
    df = df.sort_values(['enrollment_id', 'time'])
    df['time_gap'] = np.log1p(df.groupby('enrollment_id')['time'].diff().dt.total_seconds().fillna(0))
    df['token'] = df['source'] + "_" + df['event'] + "_" + df['category']
    return df

log_train = process_features(log_train)
log_test = process_features(log_test)

def build_sequences(df, truth_df):
    tokens = df.groupby("enrollment_id")["token"].apply(list)
    gaps = df.groupby("enrollment_id")["time_gap"].apply(list)
    seq_df = pd.DataFrame({"student_id": tokens.index, "tokens": tokens.values, "gaps": gaps.values})
    return seq_df.merge(truth_df, how="inner", on="student_id")

train_df = build_sequences(log_train, truth_train)
test_df = build_sequences(log_test, truth_test)

print(f"train_df: {train_df.shape}, test_df: {test_df.shape}")

In [ ]:
# ==========================================
# MERGE STATIC FSLSM FEATURES ONTO THE SEQUENCE TABLES
# ==========================================
train_df = train_df.merge(fslsm_train, on="student_id", how="left")
test_df = test_df.merge(fslsm_test, on="student_id", how="left")

missing_train = train_df[['V', 'R', 'S', 'visual_verbal_label']].isna().any(axis=1).sum()
missing_test = test_df[['V', 'R', 'S', 'visual_verbal_label']].isna().any(axis=1).sum()
print(f"Rows missing FSLSM features -> train: {missing_train}, test: {missing_test}")
# Every student in train_df/test_df comes from the same filtered logs used to build
# fslsm_train/fslsm_test, so this should be 0; fillna(0) below is just a safety net.
train_df[['V', 'R', 'S', 'visual_verbal_label']] = train_df[['V', 'R', 'S', 'visual_verbal_label']].fillna(0)
test_df[['V', 'R', 'S', 'visual_verbal_label']] = test_df[['V', 'R', 'S', 'visual_verbal_label']].fillna(0)

In [ ]:
# ==========================================
# ENCODING (source / event / category streams) — identical to 02_Data_Preprocessing.ipynb
# ==========================================
def split_token(token):
    parts = token.split("_")
    return parts if len(parts) == 3 else ["unknown", "unknown", "unknown"]

source_set, event_set, category_set = set(), set(), set()
for seq in train_df["tokens"]:
    for token in seq:
        s, e, c = split_token(token)
        source_set.add(s); event_set.add(e); category_set.add(c)

source_vocab = {v: i+1 for i, v in enumerate(source_set)}
event_vocab = {v: i+1 for i, v in enumerate(event_set)}
category_vocab = {v: i+1 for i, v in enumerate(category_set)}

def apply_encoding(df):
    def encode_multi(seq):
        src, evt, cat = [], [], []
        for token in seq:
            s, e, c = split_token(token)
            src.append(source_vocab.get(s, 0)); evt.append(event_vocab.get(e, 0)); cat.append(category_vocab.get(c, 0))
        return src, evt, cat

    encoded = df["tokens"].apply(encode_multi)
    df["src"], df["evt"], df["cat"] = encoded.str[0], encoded.str[1], encoded.str[2]
    return df

train_df, test_df = apply_encoding(train_df), apply_encoding(test_df)

In [ ]:
# ==========================================
# STANDARDIZE STATIC FSLSM FEATURES (fit on train only, applied to test)
# ==========================================
static_cols = ['V', 'R', 'S']  # visual_verbal_label is kept binary, not scaled
static_scaler = StandardScaler()
train_static_scaled = static_scaler.fit_transform(train_df[static_cols].values)
test_static_scaled = static_scaler.transform(test_df[static_cols].values)

train_static = np.concatenate([train_static_scaled, train_df[['visual_verbal_label']].values], axis=1).astype(np.float32)
test_static = np.concatenate([test_static_scaled, test_df[['visual_verbal_label']].values], axis=1).astype(np.float32)

STATIC_DIM = train_static.shape[1]
print(f"Static feature vector per learner: {static_cols + ['visual_verbal_label']} -> dim={STATIC_DIM}")

In [ ]:
# ==========================================
# VARIABLE-LENGTH SEQUENCES + aligned static feature vectors
# ==========================================
def build_variable_length(df, static_arr):
    src_list = [torch.LongTensor(s) for s in df["src"]]
    evt_list = [torch.LongTensor(e) for e in df["evt"]]
    cat_list = [torch.LongTensor(c) for c in df["cat"]]
    gap_list = [torch.FloatTensor(g) for g in df["gaps"]]
    static_list = [torch.FloatTensor(static_arr[i]) for i in range(len(df))]
    y = torch.FloatTensor(df["dropout"].values)
    ids = df["student_id"].values
    return src_list, evt_list, cat_list, gap_list, static_list, y, ids

print("Building variable-length sequences (sequential streams + static FSLSM vector)...")
src_train, evt_train, cat_train, gap_train, static_train_t, y_train, train_ids = build_variable_length(train_df, train_static)
src_test, evt_test, cat_test, gap_test, static_test_t, y_test, test_ids = build_variable_length(test_df, test_static)

lengths_train = [len(s) for s in src_train]
print(f"Train sequence lengths -> min: {min(lengths_train)}, max: {max(lengths_train)}, mean: {np.mean(lengths_train):.1f}")

## 4. Dataset / collation / batching

Extends `03_Model_Training_Evaluation.ipynb`'s `SeqDataset`, `collate_fn`, and
`BucketBatchSampler` to also carry the (fixed-length, unpadded) static feature vector
alongside each sequence.

In [ ]:
class SeqDataset(Dataset):
    def __init__(self, src, evt, cat, gaps, static, y=None):
        self.src, self.evt, self.cat, self.gaps, self.static = src, evt, cat, gaps, static
        self.y = y

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        if self.y is None:
            return self.src[idx], self.evt[idx], self.cat[idx], self.gaps[idx], self.static[idx]
        return self.src[idx], self.evt[idx], self.cat[idx], self.gaps[idx], self.static[idx], self.y[idx]


def collate_fn(batch):
    has_labels = len(batch[0]) == 6
    if has_labels:
        src, evt, cat, gaps, static, y = zip(*batch)
    else:
        src, evt, cat, gaps, static = zip(*batch)

    lengths = torch.LongTensor([len(s) for s in src])

    src = pad_sequence(src, batch_first=True, padding_value=0)
    evt = pad_sequence(evt, batch_first=True, padding_value=0)
    cat = pad_sequence(cat, batch_first=True, padding_value=0)
    gaps = pad_sequence(gaps, batch_first=True, padding_value=0.0)
    static = torch.stack(static)  # fixed-length, no padding needed

    if has_labels:
        y = torch.stack(y)
        return src, evt, cat, gaps, static, lengths, y
    return src, evt, cat, gaps, static, lengths


class BucketBatchSampler(Sampler):
    # Identical to 03_Model_Training_Evaluation.ipynb: groups similar-length
    # sequences into the same batch, then shuffles batch ORDER (not batch contents).
    def __init__(self, lengths, batch_size, shuffle=True):
        self.lengths = lengths
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __iter__(self):
        indices = np.argsort(self.lengths)
        batches = [indices[i:i+self.batch_size].tolist() for i in range(0, len(indices), self.batch_size)]
        if self.shuffle:
            np.random.shuffle(batches)
        return iter(batches)

    def __len__(self):
        return (len(self.lengths) + self.batch_size - 1) // self.batch_size


BATCH_SIZE = 128
train_lengths = [len(s) for s in src_train]
test_lengths = [len(s) for s in src_test]

train_sampler = BucketBatchSampler(train_lengths, batch_size=BATCH_SIZE, shuffle=True)
test_sampler = BucketBatchSampler(test_lengths, batch_size=BATCH_SIZE, shuffle=False)

train_loader = DataLoader(SeqDataset(src_train, evt_train, cat_train, gap_train, static_train_t, y_train),
                           batch_sampler=train_sampler, collate_fn=collate_fn)
test_loader = DataLoader(SeqDataset(src_test, evt_test, cat_test, gap_test, static_test_t, y_test),
                          batch_sampler=test_sampler, collate_fn=collate_fn)

## 5. Models

- **`BiLSTMModel`** — the sequential-only baseline, copied unchanged from
  `03_Model_Training_Evaluation.ipynb`, retrained here so it shares the exact same
  data loaders / batching / epochs as the FSLSM-augmented model.
- **`BiLSTM_FSLSM_Model`** — same BiLSTM sequential backbone, plus a small MLP over the
  4-dim static FSLSM vector (`V`, `R`, `S`, `visual_verbal_label`), fused by concatenation
  with the BiLSTM's final hidden state before the classifier head.

In [ ]:
class BiLSTMModel(nn.Module):
    # Sequential-only baseline (identical to 03_Model_Training_Evaluation.ipynb).
    def __init__(self):
        super().__init__()
        self.source_emb = nn.Embedding(len(source_vocab) + 1, 4, padding_idx=0)
        self.event_emb = nn.Embedding(len(event_vocab) + 1, 8, padding_idx=0)
        self.category_emb = nn.Embedding(len(category_vocab) + 1, 8, padding_idx=0)

        self.lstm = nn.LSTM(input_size=21, hidden_size=64, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(128, 1)

    def forward(self, src, evt, cat, gaps, lengths, static=None):
        x = torch.cat([self.source_emb(src), self.event_emb(evt), self.category_emb(cat), gaps.unsqueeze(-1)], dim=2)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        h_cat = torch.cat([h[0], h[1]], dim=1)  # (batch, 128)
        return torch.sigmoid(self.fc(h_cat)).squeeze()


class BiLSTM_FSLSM_Model(nn.Module):
    # Sequential BiLSTM backbone (same as baseline) fused with a static FSLSM
    # learning-style branch via concatenation before the classifier head.
    def __init__(self, static_dim):
        super().__init__()
        self.source_emb = nn.Embedding(len(source_vocab) + 1, 4, padding_idx=0)
        self.event_emb = nn.Embedding(len(event_vocab) + 1, 8, padding_idx=0)
        self.category_emb = nn.Embedding(len(category_vocab) + 1, 8, padding_idx=0)

        self.lstm = nn.LSTM(input_size=21, hidden_size=64, batch_first=True, bidirectional=True)

        self.static_mlp = nn.Sequential(
            nn.Linear(static_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
        )

        self.fc = nn.Linear(128 + 16, 1)

    def forward(self, src, evt, cat, gaps, lengths, static):
        x = torch.cat([self.source_emb(src), self.event_emb(evt), self.category_emb(cat), gaps.unsqueeze(-1)], dim=2)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        h_cat = torch.cat([h[0], h[1]], dim=1)  # (batch, 128)

        static_emb = self.static_mlp(static)     # (batch, 16)
        fused = torch.cat([h_cat, static_emb], dim=1)  # (batch, 144)
        return torch.sigmoid(self.fc(fused)).squeeze()


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

## 6. Shared training / evaluation loop

Both models use identical hyperparameters (3 epochs, Adam lr=1e-3, BCE loss, the same
length-bucketed batches) so the comparison below isolates the effect of the FSLSM static
feature rather than any difference in training setup.

In [ ]:
def train_and_evaluate(model, model_name, uses_static, epochs=3, lr=1e-3):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for src, evt, cat, gaps, static, lengths, labels in train_loader:
            src, evt, cat, gaps, static, labels = [t.to(device) for t in (src, evt, cat, gaps, static, labels)]

            optimizer.zero_grad()
            if uses_static:
                outputs = model(src, evt, cat, gaps, lengths, static)
            else:
                outputs = model(src, evt, cat, gaps, lengths)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"[{model_name}] Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")

    model.eval()
    preds = []
    with torch.no_grad():
        for src, evt, cat, gaps, static, lengths, _ in test_loader:
            src, evt, cat, gaps, static = [t.to(device) for t in (src, evt, cat, gaps, static)]
            if uses_static:
                outputs = model(src, evt, cat, gaps, lengths, static)
            else:
                outputs = model(src, evt, cat, gaps, lengths)
            preds.extend(outputs.cpu().numpy())

    # Predictions come out in bucketed (length-sorted) order; rebuild the matching id order.
    ordered_ids = []
    for batch in BucketBatchSampler(test_lengths, batch_size=BATCH_SIZE, shuffle=False):
        ordered_ids.extend([test_ids[i] for i in batch])

    eval_df = pd.DataFrame({"student_id": ordered_ids, "pred": preds}).merge(truth_test, on="student_id")
    eval_df["predicted_dropout"] = (eval_df["pred"] > 0.5).astype(int)

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(eval_df["dropout"], eval_df["predicted_dropout"]),
        "Precision": precision_score(eval_df["dropout"], eval_df["predicted_dropout"]),
        "Recall": recall_score(eval_df["dropout"], eval_df["predicted_dropout"]),
        "F1": f1_score(eval_df["dropout"], eval_df["predicted_dropout"]),
        "AUC": roc_auc_score(eval_df["dropout"], eval_df["pred"]),
    }
    print(f"\n[{model_name}] Test metrics:")
    for k, v in metrics.items():
        if k != "Model":
            print(f"  {k:<10}: {v:.4f}")
    return metrics, eval_df

## 7. Train the baseline (sequential-only) model

In [ ]:
torch.manual_seed(SEED)
baseline_model = BiLSTMModel()
baseline_metrics, baseline_eval_df = train_and_evaluate(
    baseline_model, model_name="Baseline BiLSTM (sequential only)", uses_static=False
)

## 8. Train the FSLSM-augmented model

In [ ]:
torch.manual_seed(SEED)
fslsm_model = BiLSTM_FSLSM_Model(static_dim=STATIC_DIM)
fslsm_metrics, fslsm_eval_df = train_and_evaluate(
    fslsm_model, model_name="BiLSTM + FSLSM static feature (concat fusion)", uses_static=True
)

## 9. Comparison table & handoff to the team

Saved to `data/results/student1_lstm_fslsm_results.csv` so Student 3 can fold this row into
the full model x fusion-approach comparison table (Task 4).

In [ ]:
comparison_df = pd.DataFrame([baseline_metrics, fslsm_metrics])
comparison_df["Fusion Approach"] = ["N/A (no static features)", "Concatenation (intermediate fusion)"]
comparison_df["Input Features"] = [
    "Sequential (source/event/category tokens + time gaps)",
    "Sequential + static FSLSM Visual-Verbal score (V, R, S, binary label)"
]
comparison_df = comparison_df[["Model", "Fusion Approach", "Input Features", "Accuracy", "Precision", "Recall", "F1", "AUC"]]

print(comparison_df.to_string(index=False))

improvement = fslsm_metrics["AUC"] - baseline_metrics["AUC"]
print(f"\nAUC change from adding the FSLSM static feature: {improvement:+.4f}")

os.makedirs("data/results", exist_ok=True)
comparison_df.to_csv("data/results/student1_lstm_fslsm_results.csv", index=False)
print("Saved comparison table to data/results/student1_lstm_fslsm_results.csv")

## Notes for the team

- `data/processed/fslsm_train.csv` and `data/processed/fslsm_test.csv` contain the raw
  `V`, `R`, `S`, `visual_verbal_label` per `student_id` for the first-2-week window — reuse
  these directly rather than recomputing, so every experiment scores learning style the
  same way.
- The fusion used here (concatenate a static-feature MLP embedding with the BiLSTM's final
  hidden state) is one reasonable baseline integration. Student 3's fusion-comparison
  notebook should treat this as the "intermediate/concatenation fusion" row and add
  early-fusion (concatenate static features onto every timestep's input), late-fusion
  (separate classifiers combined by a weighted/learned combiner), and attention-based
  fusion (e.g. using the static embedding to compute attention over the sequence) as
  additional rows against the *same* train/test split and metrics used here.
- If AUC does not improve over the baseline, that is itself a valid, reportable finding —
  it would suggest the 2-week FSLSM signal is largely redundant with what the raw
  event-count/category tokens already give the BiLSTM.